# AirGuard AI: Exploratory Data Analysis & Machine Learning Pipeline
### An End-to-End Regression Study for Urban Air Quality Index (AQI) Forecasting

**Author**: AirGuard AI Engineering Team  
**Objective**: Build and evaluate production-grade regression models to predict next-day AQI using multi-city atmospheric and meteorological observations.

---
### Table of Contents:
1. **Dataset Overview & Schema Validation**
2. **Data Cleaning & Missing Value Handling**
3. **Descriptive Statistics & Summary Metrics**
4. **Correlation Analysis & Multicollinearity Inspection**
5. **AQI Distribution & Categorization**
6. **Pollutant Relationships (PM2.5, PM10, Gaseous vs AQI)**
7. **Time-Series Trends & Seasonality**
8. **Feature Engineering (Lags, Rolling Averages, Cyclical Encodings)**
9. **Temporal Train/Test Split (Preventing Data Leakage)**
10. **Model Training & Comparison (Baseline Ridge vs Random Forest vs Gradient Boosting)**
11. **Evaluation Metrics (MAE, RMSE, R²)**
12. **Feature Importance & Interpretability**
13. **Interview Takeaways & Production Deployment Strategy**


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Load Dataset
data_path = os.path.join("..", "data", "air_quality_cleaned.csv")
df = pd.read_csv(data_path)
df['time'] = pd.to_datetime(df['time'])
print(f"Dataset Shape: {df.shape}")
print(f"Unique Cities: {df['city_name'].unique().tolist()}")
df.head()


## 2. Data Cleaning & Missing Value Assessment
In time-series environmental data, sensor downtime or network packet drops can lead to missing measurements.
We perform backward/forward imputation bounded by temporal proximity, avoiding synthetic distortions.


In [ ]:
# Check for any missing values across features
missing_summary = df.isnull().sum()
print("Missing values per column:")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "Zero missing values (dataset pre-cleaned).")


## 3. Descriptive Statistics
Examine the central tendencies, spread, and physical plausibility of particulate and meteorological features.


In [ ]:
pollutants = ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed', 'calculated_aqi']
df[pollutants].describe().T[['mean', 'std', 'min', '50%', 'max']]


## 4. Correlation Analysis & Feature Interactions
Understanding the linear correlation between pollutants and AQI helps prioritize feature subsets.


In [ ]:
corr_matrix = df[pollutants].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, cbar=True)
plt.title("Pearson Correlation Heatmap: Atmospheric & Weather Variables vs AQI")
plt.tight_layout()
plt.show()


## 5. AQI Distribution & Category Spread
We examine whether the target variable exhibits heavy tails or right-skewness, typical of severe pollution episodes.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['calculated_aqi'], kde=True, ax=axes[0], color='#3b82f6', bins=30)
axes[0].set_title("Distribution of Air Quality Index (AQI)")
axes[0].set_xlabel("Calculated AQI")

category_counts = df['calculated_category'].value_counts()
axes[1].pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', startangle=140)
axes[1].set_title("AQI Category Proportion")

plt.tight_layout()
plt.show()


## 6. Feature Engineering: Lags, Rolling Averages & Cyclical Encodings
Environmental processes are strongly auto-correlated (today's PM2.5 heavily influences tomorrow's) and follow diurnal (hourly) and seasonal (monthly) patterns.

**Engineered Features**:
- **Cyclical hour**: $\sin(2\pi \cdot \text{hour} / 24)$ and $\cos(2\pi \cdot \text{hour} / 24)$
- **Lagged variables**: 1-hour and 24-hour lag of PM2.5, PM10
- **Rolling statistics**: 6-hour and 24-hour moving averages
- **Target**: AQI shifted 24 hours into the future ($t + 24$)


In [ ]:
def create_features(data):
    df_feat = data.copy().sort_values(['city_name', 'time']).reset_index(drop=True)
    
    # Cyclical
    h = df_feat['time'].dt.hour
    df_feat['hour_sin'] = np.sin(2 * np.pi * h / 24.0)
    df_feat['hour_cos'] = np.cos(2 * np.pi * h / 24.0)
    df_feat['day_of_week'] = df_feat['time'].dt.dayofweek
    df_feat['month'] = df_feat['time'].dt.month
    df_feat['is_weekend'] = (df_feat['day_of_week'] >= 5).astype(int)
    
    # Grouped lags & rollings
    grp = df_feat.groupby('city_name')
    df_feat['pm25_lag1'] = grp['pm25'].shift(1)
    df_feat['pm25_lag24'] = grp['pm25'].shift(24)
    df_feat['pm10_lag1'] = grp['pm10'].shift(1)
    df_feat['pm25_roll_mean_6h'] = grp['pm25'].transform(lambda s: s.shift(1).rolling(6, min_periods=1).mean())
    df_feat['pm25_roll_mean_24h'] = grp['pm25'].transform(lambda s: s.shift(1).rolling(24, min_periods=1).mean())
    df_feat['aqi_roll_mean_24h'] = grp['calculated_aqi'].transform(lambda s: s.shift(1).rolling(24, min_periods=1).mean())
    df_feat['wind_roll_mean_6h'] = grp['wind_speed'].transform(lambda s: s.shift(1).rolling(6, min_periods=1).mean())
    
    # Target
    df_feat['target_aqi_24h'] = grp['calculated_aqi'].shift(-24)
    
    features = [
        "pm25", "pm10", "no2", "so2", "co", "o3",
        "temperature", "humidity", "wind_speed",
        "hour_sin", "hour_cos", "day_of_week", "month", "is_weekend",
        "pm25_lag1", "pm25_lag24", "pm10_lag1",
        "pm25_roll_mean_6h", "pm25_roll_mean_24h",
        "aqi_roll_mean_24h", "wind_roll_mean_6h"
    ]
    df_clean = df_feat.dropna(subset=features + ['target_aqi_24h']).reset_index(drop=True)
    return df_clean, features

featured_df, feature_cols = create_features(df)
print(f"Dataset with features: {featured_df.shape}")
print(f"Total features: {len(feature_cols)}")


## 7. Strict Chronological Train/Test Split (No Data Leakage)
A critical requirement in time-series: **Never use random k-fold or shuffle-based splitting**, because future records will leak into the past.
We split the earliest 80% chronologically for training and evaluate on the final 20% future data.


In [ ]:
train_subsets, test_subsets = [], []

for city, city_df in featured_df.groupby('city_name'):
    city_sorted = city_df.sort_values('time').reset_index(drop=True)
    split_pt = int(len(city_sorted) * 0.8)
    train_subsets.append(city_sorted.iloc[:split_pt])
    test_subsets.append(city_sorted.iloc[split_pt:])

train_df = pd.concat(train_subsets, ignore_index=True)
test_df = pd.concat(test_subsets, ignore_index=True)

X_train, y_train = train_df[feature_cols], train_df['target_aqi_24h']
X_test, y_test = test_df[feature_cols], test_df['target_aqi_24h']

print(f"X_train samples: {len(X_train):,} | X_test samples: {len(X_test):,}")


## 8. Model Training, Evaluation & Comparison
We compare three distinct architectural paradigms:
1. **Baseline**: Regularized Linear (Ridge Regression)
2. **Bagging**: Random Forest Regressor
3. **Boosting**: Gradient Tree Boosting


In [ ]:
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

models = {
    "Baseline (Ridge)": (Ridge(alpha=1.0), True),
    "Random Forest": (RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1), False),
    "Gradient Boosting": (GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=5, random_state=42), False)
}

comparison_metrics = []

for name, (clf, use_scale) in models.items():
    X_tr = X_tr_sc if use_scale else X_train
    X_te = X_te_sc if use_scale else X_test
    
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    comparison_metrics.append({
        "Model": name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "R2 Score": round(r2, 4)
    })

comparison_df = pd.DataFrame(comparison_metrics)
display(comparison_df)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.barplot(data=comparison_df, x="Model", y="MAE", ax=axes[0], palette="Blues_d")
axes[0].set_title("Mean Absolute Error (Lower is better)")

sns.barplot(data=comparison_df, x="Model", y="RMSE", ax=axes[1], palette="Oranges_d")
axes[1].set_title("Root Mean Squared Error (Lower is better)")

sns.barplot(data=comparison_df, x="Model", y="R2 Score", ax=axes[2], palette="Greens_d")
axes[2].set_title("R² Score (Higher is better)")

plt.tight_layout()
plt.show()


## 10. Explainability: Feature Importance Analysis
Which atmospheric indicators have the strongest influence on the 24-hour ahead AQI prediction?


In [ ]:
gb_model = models["Gradient Boosting"][0]
importances = gb_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance (%)': importances * 100
}).sort_values('Importance (%)', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df.head(10), x='Importance (%)', y='Feature', palette='viridis')
plt.title("Top 10 Influential Features for 24h AQI Forecasting (Gradient Boosting)")
plt.xlabel("Relative Contribution (%)")
plt.tight_layout()
plt.show()


## 11. Placement Interview Defense Summary

1. **Why Regression instead of Classification?**
   AQI is an intrinsically continuous index (0 to 500+). Regressing the exact numerical value preserves ordinal magnitude and allows computing continuous prediction intervals; category mappings are derived post-prediction without loss of information.

2. **How was Data Leakage Prevented?**
   - No forward-looking rolling windows: all moving averages use `shift(1)`.
   - Chronological train/test split: models are trained on past observations ($t \le T_0$) and tested on strictly unseen future time horizons ($t > T_0$).

3. **Why Gradient Boosting / Random Forest vs Linear Baseline?**
   Atmospheric dispersion exhibits non-linear threshold effects (e.g. wind velocity drastically cleanses pollutants above 12 km/h; thermal inversions trap PM2.5 in low temperatures). Ensemble tree models capture non-linear feature interactions without manual polynomial expansion.

4. **Production Serving**:
   Serialized via `joblib`, deployed via Python FastAPI microservice, orchestrated by Spring Boot backend, and presented through React dashboard.
